# Nomos Master Pipeline: Data to Regimes

This notebook contains the unified pipeline for Project Nomos:
1. **Data Ingestion** (Yahoo & Kite)
2. **Feature Engineering** (Log-Returns, VIX Spreads, Volatility Detrending)
3. **Stationarity Enforcement** (Iterative Differencing & Cleaning)
4. **Regime Detection** (HMM Training & Visualization)

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.append(os.path.abspath('..'))

from src.data.data_manager import DataManager
from src.models.hmm_model import NomosHMM

# Global Styling
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (15, 8)

## 1. Full Data Pipeline
Fetching and processing all features in one go.

In [ ]:
manager = DataManager("../config/config.yaml")

print("Step 1: Fetching Raw Data...")
df_raw = manager.fetch_all_data(start_date="2008-01-01")

print("Step 2: Engineering Features & Enforcing Stationarity...")
df_features = manager.process_data(df_raw)

print("Step 3: Scaling Features (Z-Score)...")
df_final = manager.scale_features(df_features)

print("\nDiagnostics Report:")
display(manager.check_features(df_features))

print(f"\nFinal shape: {df_final.shape}")
display(df_final.head())

## 2. Train Regime Detection (HMM)
Identifying market states.

In [ ]:
# Select engineered features
feature_cols = ['NIFTY50_Ret', 'VIX_Spread', 'Corr_NIFTY50_Gold', 'Detrended_Vol']

# Train HMM
nomos_hmm = NomosHMM(n_components=3)
nomos_hmm.fit(df_final, feature_cols)

print("Model Converged! State Labels:", nomos_hmm.state_labels)
nomos_hmm.save_model("../models/latest_regime_model.joblib")

## 3. Visualization
Mapping the 'Shadow' states to the NIFTY chart.

In [ ]:
# Predict regimes
res_df = df_final.copy()
res_df['Regime'] = nomos_hmm.predict(df_final)

# Create the overlay plot using Cumulative Returns as a proxy for the price path
colors = {'Bull': '#2ecc71', 'Neutral': '#95a5a6', 'Bear': '#e74c3c'}

plt.figure(figsize=(15, 8))
for label, color in colors.items():
    mask = res_df['Regime'] == label
    if mask.any():
        plt.scatter(res_df.index[mask], res_df.loc[mask, 'NIFTY50_Ret'].cumsum(), 
                    label=label, c=color, s=15, alpha=0.7)

plt.title("NIFTY50 Regime Map (HMM Trained on Multi-Asset Features)", fontsize=16)
plt.ylabel("Cumulative Log-Returns", fontsize=12)
plt.legend(title="Market Regime")
plt.show()

## 4. Transition Stability
Probability of jumping between regimes.

In [ ]:
labels = [nomos_hmm.state_labels[i] for i in range(3)]
trans_mat = pd.DataFrame(nomos_hmm.model.transmat_, index=labels, columns=labels)

plt.figure(figsize=(8, 6))
sns.heatmap(trans_mat, annot=True, cmap="RdYlGn", fmt=".2f")
plt.title("Regime Transition Matrix (State Persistence)")
plt.show()